# 05 — Selection readout

In the TIL co-culture arm, cells are being killed. A perturbation's
*representation* in that arm relative to control is therefore a phenotype in
its own right — the classic pooled-screen fitness readout — and it costs
almost nothing to compute from data already loaded.

**Why this belongs in this project and not a separate one:** transcriptional
response and survival are not the same phenotype. A screen that measures only
one is blind to the other, and the disagreements between them are the finding.

**Caveat, stated up front:** this is not a properly timecourse-controlled
dropout screen. Differential representation can also reflect infection
efficiency or proliferation differences unrelated to immune pressure. The
control arm partially handles this. Say so rather than overclaiming.

In [ ]:
# =============================================================================
# nb05 — Condition-stratified differential expression
#
# Produces the core object of the project: a log2FC signature for every
# (perturbation, condition) pair, each computed against CONDITION-MATCHED
# control-guide cells. Everything in the context-dependence analysis reads
# from this matrix.
#
# THE DESIGN CONSTRAINT. Every contrast is perturbation vs control-guide cells
# IN THE SAME CONDITION. Controls are never pooled across conditions: IFN-γ
# and TIL co-culture shift the baseline transcriptome enormously, and pooling
# would attribute that shift to every perturbation, producing a screen in which
# everything is a hit and none of it means anything.
#
# REPLICATE STRUCTURE. sgRNAs. The library carries 3 guides per target, and
# those are genuinely independent perturbation events — different cut sites,
# different off-target profiles, independently infected cells. Splitting cells
# at random would give the model a variance term reflecting only sampling
# noise; splitting by guide gives it something closer to real experimental
# variance. Eligibility was gated in nb02: a contrast needs >=2 guides clearing
# the cell-count minimum in that condition, since one guide leaves no
# within-group variance to estimate.
#
# WHAT THE OUTPUT IS AND IS NOT. Three guides, one biological sample, no
# true replication. DE p-values are anti-conservative and the ranking should be
# read as a ranking. Permutation-tested effect sizes come later.
# =============================================================================

%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig
from src.pseudobulk import make_pseudobulk
from src.stats import signature_matrix

cfg    = load_config()
panels = load_panels()
P      = paths(cfg)
SEED   = set_seed(cfg)
apply_style(cfg)
sc.settings.verbosity = 1

s          = cfg["schema"]["obs"]
PERT       = s["perturbation"]
COND       = s["condition"]
GUIDE      = s["guide"]
CTRL       = cfg["schema"]["control_label"]
REF        = cfg["schema"]["conditions"]["reference"]
cond_order = ["Control", "IFNγ", "Co-culture"]
pal        = condition_palette(cfg)

# ---- data -----------------------------------------------------------------
import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna, adt = mdata["rna"], mdata["adt"]

# ---- carried forward from earlier notebooks -------------------------------
elig  = pd.read_csv(P.tables / "02_guide_eligibility.csv", index_col=0)   # nb02
flags = pd.read_csv(P.tables / "02_gene_flags.csv", index_col=0)          # nb02
emb   = pd.read_parquet(P.data_interim / "03_embedding.parquet")          # nb03

# nb03 identified cluster 12 as contaminating T cells — lymphocytes that
# acquired a guide through ambient RNA or doublet formation and so survived
# the MOI=1 filter. They are not melanoma cells and must not enter any
# perturbation contrast.
tcell_cells = emb.index[emb["cluster"].astype(str) == "12"]
keep = ~rna.obs_names.isin(tcell_cells)
print(f"removing {(~keep).sum():,} contaminating lymphocytes")
rna = rna[keep].copy()
adt = adt[rna.obs_names].copy()

print(f"\nRNA: {rna.n_obs:,} cells x {rna.n_vars:,} genes")
print(f"ADT: {adt.n_obs:,} cells x {adt.n_vars} features")
print(f"\ncells per condition:")
print(rna.obs[COND].value_counts().reindex(cond_order).to_string())

testable = (elig >= cfg["de"].get("min_guides_per_contrast", 2))
print(f"\ntestable contrasts: {testable.values.sum()} / {testable.size}")
print(f"genes testable in all 3 conditions: {testable.all(axis=1).sum()} / {len(elig)}")
print(f"high guide-spread flags: {flags['high_spread'].sum()}")

assert set(rna.obs["MOI"].unique()) == {1}

In [ ]:
# ---- gene universe for DE -------------------------------------------------
# Wider than the 2,000 HVGs used for the embedding. There, the goal was the
# dominant axes of variation; here, a gene excluded from the set can never
# appear as a perturbation effect no matter how strongly it moves. The cost of
# a wider set is multiple-testing burden — but the p-values are already soft
# given the replication structure, so blind spots are the worse failure.
#
# Still selected on control-guide cells only: choosing genes on all cells
# would select partly on the perturbation effects being measured.
N_GENES = 5000

rna_n = rna.copy()
rna_n.X = rna_n.layers["counts"].copy()
sc.pp.normalize_total(rna_n, target_sum=1e4)
sc.pp.log1p(rna_n)

ctrl_cells = rna_n.obs[PERT].astype(str) == CTRL
print(f"control-guide cells: {ctrl_cells.sum():,}")

ctrl_sub = rna_n[ctrl_cells].copy()
sc.pp.highly_variable_genes(ctrl_sub, n_top_genes=N_GENES,
                            flavor="seurat_v3", layer="counts")
hvg = ctrl_sub.var_names[ctrl_sub.var["highly_variable"]].tolist()
print(f"genes selected: {len(hvg)}")

# sanity: the pathways with a strong prior in this system should be present
hvg_set = set(hvg)
for name, genes in panels["gene_sets"].items():
    if isinstance(genes, list):
        hit = hvg_set & set(genes)
        print(f"  {name:28s} {len(hit):2d}/{len(genes)}")

In [ ]:
# ---- pseudobulk, keyed on perturbation | condition | guide -----------------
# Guides are the replicate unit, so each (perturbation, condition) group is
# split into its 3 constituent guides rather than into random subsets. Control
# cells are split the same way — the control guide pool is large (75 guides),
# so those are binned into 3 pseudo-guides to give a comparable replicate
# count without one control group dwarfing the design.
#
# Raw counts are SUMMED, then modelled. Summing counts rather than averaging
# normalised values is what makes the result valid input to a negative-binomial
# model: it treats the group as one deep pseudo-sample rather than weighting a
# 500-UMI cell equally with an 8,000-UMI one.
MIN_CELLS_PER_GUIDE = cfg["de"].get("min_cells_per_guide", 10)
N_CTRL_BINS = 3

rng = np.random.default_rng(SEED)

pert_v  = rna.obs[PERT].astype(str).values
cond_v  = rna.obs[COND].astype(str).values
guide_v = rna.obs[GUIDE].astype(str).values

# control cells get synthetic replicate labels; targeted cells keep their guide
is_ctrl = pert_v == CTRL
rep_v = guide_v.copy()
rep_v[is_ctrl] = [f"{CTRL}_bin{b}" for b in rng.integers(0, N_CTRL_BINS, is_ctrl.sum())]

keys = np.array([f"{p}|{c}|{r}" for p, c, r in zip(pert_v, cond_v, rep_v)])

counts_mat = rna.layers["counts"] if "counts" in rna.layers else rna.X
gene_ix = pd.Index(rna.var_names)
col_keep = gene_ix.get_indexer(hvg)

uniq = pd.unique(keys)
row_of = {k: i for i, k in enumerate(uniq)}
M = sp.csr_matrix(
    (np.ones(len(keys)), ([row_of[k] for k in keys], np.arange(len(keys)))),
    shape=(len(uniq), rna.n_obs),
)
pb = M @ counts_mat
pb = np.asarray(pb.todense()) if sp.issparse(pb) else np.asarray(pb)

counts = pd.DataFrame(np.rint(pb[:, col_keep]).astype(int),
                      index=uniq, columns=hvg)

parts = pd.Series(uniq).str.split("|", expand=True)
meta = pd.DataFrame({"perturbation": parts[0].values,
                     "condition":    parts[1].values,
                     "replicate":    parts[2].values}, index=uniq)
meta["n_cells"] = pd.Series(keys).value_counts().reindex(uniq).values

# drop thin replicates — a guide with 8 cells gives a noisy profile that reads
# as biological variance when it is sampling noise
before = len(meta)
keep_rep = meta["n_cells"] >= MIN_CELLS_PER_GUIDE
counts, meta = counts.loc[keep_rep], meta.loc[keep_rep]
print(f"pseudobulk groups: {before} -> {len(meta)} "
      f"(>= {MIN_CELLS_PER_GUIDE} cells)")

# a contrast needs >=2 replicates on BOTH sides to have any within-group variance
rep_n = (meta.groupby(["perturbation", "condition"], observed=True)
         .size().unstack(fill_value=0).reindex(columns=cond_order, fill_value=0))
print(f"\nreplicates per contrast:")
print(rep_n.apply(pd.Series.value_counts).fillna(0).astype(int).to_string())
print(f"\ncontrol replicates: {rep_n.loc[CTRL].to_dict()}")

eligible = rep_n.drop(index=CTRL, errors="ignore") >= 2
print(f"\neligible contrasts: {eligible.values.sum()} / {eligible.size}")
print(f"genes eligible in all 3 conditions: {eligible.all(axis=1).sum()} / {len(eligible)}")

In [ ]:
print(f"perturbations in filtered data: {rna.obs[PERT].nunique() - 1}")
print(f"reaching pseudobulk: {len(rep_n) - 1}")

In [ ]:
# which genes don't reach pseudobulk (n=2 replicates in both comparisons)

in_data = set(rna.obs[PERT].unique()) - {CTRL}
in_pseudo = set(rep_n.index) - {CTRL}
dropped = in_data - in_pseudo
print(f"dropped ({len(dropped)}):")
for g in sorted(dropped):
    # show why: how many cells in each condition
    cells = rna.obs[rna.obs[PERT] == g][COND].value_counts().reindex(cond_order, fill_value=0)
    print(f"  {g:12s}  {cells.to_dict()}")

Eleven targets dropped between cell filtering and the eligibility gate, all
housekeeping or essential genes (AHCY, ATP1A1, CCT6A, DNAJC9, FARSA, PSMA7,
RACK1, SNRPE, SNRPF, TUBB, UBL5). Each had 2–15 cells per condition, consistent
with essential-gene knockouts being poorly tolerated — cells carrying those guides
likely died before capture, so representation was below the per-guide cell minimum
in at least one condition. None are immune-relevant targets whose loss would affect
interpretation of the screen.

These CRISPR-KO won't be included in DE testing.

In [ ]:
# ---- DE: 667 contrasts, perturbation vs condition-matched control ----------
# One DESeq2 fit per (perturbation, condition). Fitting each contrast
# separately rather than one global model with an interaction term keeps every
# comparison against its own condition's controls, which is the design
# constraint the project rests on.
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from tqdm.auto import tqdm
import time

MIN_REPS = 2
de_results = {}
failed = []
t0 = time.time()

jobs = [(p, c) for c in cond_order
        for p in eligible.index[eligible[c]]]
print(f"running {len(jobs)} contrasts on {len(hvg)} genes")

for pert, cond in tqdm(jobs):
    sub = meta[meta["condition"] == cond]
    idx = sub.index[sub["perturbation"].isin([pert, CTRL])]
    md_ = sub.loc[idx, ["perturbation"]].copy()

    # force control as reference level so DESeq2 coefficients are predictable
    md_["perturbation"] = pd.Categorical(
        md_["perturbation"],
        categories=[CTRL, pert],   # CTRL first = reference = intercept
        ordered=False
    )

    if md_["perturbation"].value_counts().min() < MIN_REPS:
        failed.append((pert, cond, "too few replicates"))
        continue

    try:
        dds = DeseqDataSet(
            counts=counts.loc[idx],
            metadata=md_,
            design="~perturbation",
            refit_cooks=True,
            quiet=True,
        )
        dds.deseq2()
        st = DeseqStats(dds, contrast=["perturbation", pert, CTRL], quiet=True)
        st.summary()
        if cfg["de"]["lfc_shrink"]:
            st.lfc_shrink(coeff=f"perturbation[T.{pert}]")
        de_results[(pert, cond)] = st.results_df
    except Exception as e:
        failed.append((pert, cond, str(e)[:80]))

print(f"\ndone in {(time.time()-t0)/60:.1f} min")
print(f"succeeded: {len(de_results)} | failed: {len(failed)}")
if failed:
    print(pd.DataFrame(failed, columns=["pert", "cond", "reason"])
          .head(10).to_string(index=False))

In [ ]:
# ---- assemble and persist the signature matrix ----------------------------
sig = signature_matrix(de_results, value_col="log2FoldChange", genes=hvg)
sig.to_parquet(P.data_processed / "05_signatures.parquet")

padj = signature_matrix(de_results, value_col="padj", genes=hvg)
padj.to_parquet(P.data_processed / "05_padj.parquet")

print(f"signature matrix: {sig.shape[0]} contrasts x {sig.shape[1]} genes")
print(sig.index.get_level_values('condition').value_counts().to_string())

In [ ]:
# for each perturbation, look up its own gene's log2FC in its own rows
# should be negative for most — NMD from frameshift
self_check = []
for pert in sig.index.get_level_values("perturbation").unique():
    if pert not in sig.columns:
        continue
    rows = sig.loc[pert]
    for cond in rows.index:
        self_check.append({
            "perturbation": pert,
            "condition": cond,
            "self_lfc": rows.loc[cond, pert]
        })

sc_df = pd.DataFrame(self_check)
print(f"median self-LFC: {sc_df['self_lfc'].median():.3f}")
print(f"fraction negative: {(sc_df['self_lfc'] < 0).mean():.1%}")
print(sc_df.groupby("condition")["self_lfc"].median().round(3).to_string())

In [ ]:
# B2M and JAK1 should have strong self-knockdown under IFNg
for gene in ["B2M", "JAK1", "JAK2", "STAT1", "CD58"]:
    if gene in sig.columns and gene in sig.index.get_level_values("perturbation"):
        row = sig.loc[gene, "self_lfc"] if "self_lfc" in sig.columns else None
        vals = sig.loc[gene][gene] if gene in sig.index.get_level_values("perturbation") else None
        print(f"{gene}: {sig.loc[gene, gene].to_dict()}")

In [ ]:
# ---- persist the signature matrix -----------------------------------------
sig.to_parquet(P.data_processed / "05_signatures_lfc.parquet")
padj.to_parquet(P.data_processed / "05_signatures_padj.parquet")

# save the de_results dict as individual CSVs in case we need per-gene stats
de_dir = P.data_processed / "05_de_results"
de_dir.mkdir(exist_ok=True)
for (pert, cond), df in de_results.items():
    df.to_csv(de_dir / f"{pert}__{cond}.csv")

print(f"signature matrix:  {sig.shape}  -> 05_signatures_lfc.parquet")
print(f"padj matrix:       {padj.shape} -> 05_signatures_padj.parquet")
print(f"per-contrast CSVs: {len(de_results)} files -> 05_de_results/")

import session_info
session_info.show()